# Chinese Data Analysis

This notebook is part of the English edition of the Big Data Analysis course materials.

This notebook introduces a China-focused dataset from the National Bureau of Statistics of China (NBS), shows how to clean a province-level spreadsheet, merge it with a panel dataset, and construct simple derived variables.

A short English variable dictionary is included in `data/china/china_panel_data_dictionary.csv`.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "02_chinese_data_analysis"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

## Goals: 
### 1. Familiarize with data from the National Bureau of Statistics of China and reshape it into panel data.
### 2. Merge two datasets.
### 3. Make some simple calculations for economics students.

## The National Bureau of Statistics of China provides plenty of Chinese social and economic data.
## https://data.stats.gov.cn/easyquery.htm?cn=C01

In [ ]:
import pandas as pd

# Q1 How to read an Excel file starting from the fifth row?
## To read xlsx format files, you need to install openpyxl first in order to use read_excel to read the data.

In [ ]:
%pip install -q openpyxl

In [ ]:
data1 = pd.read_excel(
    DATA_DIR / "china" / "china_corporate_units_by_region_2020.xlsx",
    sheet_name=0,
    header=5,
)
data1.head(10)

# Q2 How to rename columns？

In [ ]:
### The first few columns need clearer names.
### The remaining columns contain line breaks, so we remove text after each line break.

data1 = data1.rename(columns={
    "Unnamed: 0": "region_cn",
    "Unnamed: 1": "region_en",
    "Unnamed: 2": "corporate_units",
}).rename(columns=lambda x: x.split("\n")[0] if isinstance(x, str) else x)

data1

# The first column contains region names, and many values contain unnecessary spaces.
# Q3. How can we use `str.replace()` to remove spaces from `region_cn`?

In [ ]:
data1["region_cn"] = data1["region_cn"].str.replace(" ", "")
data1["region_cn"].unique()

In [ ]:
data1.head()

# The table contains region names and variables, but it does not yet contain a year column.
# Q4. How can we generate a column with a constant value?

In [ ]:
### If you generate a numeric column:
data1["year"] = 2020

### If you generate a string column:
# data1["source_label"] = "NBS"

data1

# Let's merge this table with a complete panel dataset for additional calculations.

See `data/china/china_panel_data_dictionary.csv` for English aliases of the Chinese variables used below.

# Q5: How to merge two dataset?
## https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

In [ ]:
### Read the panel data and merge it with `data1`.
df = pd.read_excel(
    DATA_DIR / "china" / "china_panel_data.xlsx",
    sheet_name=0,
).rename(columns={"地区": "region_cn", "年份": "year"}).merge(data1, how="left", on=["region_cn", "year"])

df.head()

# Now, let's perform some simple calculations!

In [ ]:
### First, check if there are any missing values.
df.isna().sum().sort_values(ascending=False)

# Q6. How can we create new columns through addition, subtraction, multiplication, division, and logarithms for later analysis?

In [ ]:
df["import_export"] = df["货物出口金额（万美元）"] + df["货物进口金额（万美元）"]

df["pop_rural"] = df["总人口（万人）"] - df["城镇人口（万人）"]

df["rd_dig"] = df["R&D经费（万元）"] * df["数字普惠金融指数"]

df["internet"] = df["互联网宽带接入用户（万户）"] / df["总人口（万人）"]

import numpy as np
df["lngdp"] = np.log(df["国内生产总值（当年价）（亿元）"])

df.head()

In [ ]:
df.columns

# Q7. How can we calculate the growth rate of a specific column?

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html


In [ ]:
df["rd_growth_rate"] = df.groupby("region_cn")["R&D经费（万元）"].pct_change()
df.head()

In [ ]:
df.query("year == 2013")

# Q8. How can we calculate the cumulative sum?

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.cumsum.html

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html


In [ ]:
df["cumulative_ecommerce_sales"] = df.groupby("region_cn")["电子商务销售额（亿元）"].cumsum()
df.head()

# Q9 How to assign codes to each province?

In [ ]:
### Read the administrative codebook.

### The `行政区划代码` column is stored as a float in the original file, which introduces decimal points.
### For easier processing, convert it to integer first and then to string.

code = pd.read_excel(
    DATA_DIR / "china" / "china_administrative_division_codebook.xlsx",
    na_values="..",
    sheet_name=0,
    header=2,
    usecols="B:C",
).dropna().astype({"行政区划代码": int}).astype({"行政区划代码": str}).rename(
    columns={"行政区划代码": "admin_code", "单位名称": "region_cn"}
)

code.head()

## Q9.1. How can we keep only province-level codes rather than city- or district-level codes?

### Provinces, cities, and districts all have their own administrative codes.
### For example, Beijing is `110000`, while Dongcheng and Xicheng are `110101` and `110102`.
### Province-level codes end with `0000`, so we filter for that pattern.

In [ ]:
code = code[code["admin_code"].str[-4:] == "0000"]
code.head()

## Q9.2. How can we remove suffixes such as `省`, `市`, and `自治区` from the region names so they match the panel data?

In [ ]:
code["region_cn"] = code["region_cn"].str.rstrip("市省自治区壮族回族维吾尔特别行政区")
code

# Merge the codebook with the panel dataset so that we get a cleaner province-level panel with administrative codes.

In [ ]:
df_final = df.merge(code, how="left", on=["region_cn"])
df_final

# We obtained a clean panel dataset from the National Bureau of Statistics of China and learned how to merge datasets, perform simple calculations, rename columns, handle strings, and attach administrative codes.